In [ ]:
import json
import pandas as pd

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


def resolver_base() -> Path:
    """Resolve o diretório raiz do projeto em qualquer ambiente."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tsv").exists() and (candidate / "catalogos").exists():
            return candidate

    return cwd


BASE = resolver_base()
DIR_CATALOGOS = BASE / "catalogos"

if not DIR_CATALOGOS.exists():
    raise FileNotFoundError(f"Diretório de catálogos não encontrado: {DIR_CATALOGOS}")

ARQUIVO_REVISAO = DIR_CATALOGOS / "revisao_cursos.xlsx"

Mounted at /content/drive


In [2]:
# ============================================================
# CARREGAMENTO E VALIDAÇÃO
# ============================================================

df = pd.read_excel(ARQUIVO_REVISAO)

df.columns = (
    df.columns
      .astype(str)
      .str.strip()
)

COLUNAS = [
    "Curso histórico",
    "Curso canônico",
    "Ignorar"
]

faltantes = [
    c for c in COLUNAS
    if c not in df.columns
]

if faltantes:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltantes}")

df["Curso histórico"] = (
    df["Curso histórico"]
      .fillna("")
      .astype(str)
      .str.strip()
)

df["Curso canônico"] = (
    df["Curso canônico"]
      .fillna("")
      .astype(str)
      .str.strip()
)

df["Ignorar"] = (
    df["Ignorar"]
      .fillna("")
      .astype(str)
      .str.upper()
      .str.strip()
)

# ============================================================
# DUPLICIDADES
# ============================================================

ativos = df[df["Ignorar"] != "SIM"].copy()

duplicados = ativos[
    ativos["Curso histórico"].duplicated(keep=False)
]

if not duplicados.empty:

    print("Cursos históricos duplicados:")

    display(
        duplicados.sort_values("Curso histórico")
    )

    raise ValueError(
        "Existem cursos históricos duplicados."
    )

# IGNORAR INCONSISTÊNCIAS
ignorar_inconsistente = df[
    (df["Ignorar"] == "SIM")
    &
    (df["Curso canônico"] != "")
]

if not ignorar_inconsistente.empty:

    display(ignorar_inconsistente)

    raise ValueError(
        "Existem cursos marcados para ignorar com curso canônico preenchido."
    )

# Pendências

pendentes = df[
    (df["Ignorar"] != "SIM")
    &
    (df["Curso canônico"] == "")
]

if not pendentes.empty:
    display(pendentes)
    raise ValueError(
        f"{len(pendentes)} cursos ainda sem definição."
    )

print(f"✓ {len(df)} registros validados.")

✓ 136 registros validados.


In [3]:
# ============================================================
# CONSOLIDAÇÃO
# ============================================================

catalogo = {}
indice = {}

for _, linha in df.iterrows():

    if linha["Ignorar"] == "SIM":
        continue

    historico = linha["Curso histórico"]
    canonico = linha["Curso canônico"]

    catalogo.setdefault(canonico, []).append(historico)

    indice[historico] = canonico

catalogo = {
    curso: sorted(set(variantes))
    for curso, variantes in sorted(catalogo.items())
}

indice = dict(sorted(indice.items()))

print("=" * 70)
print("CATÁLOGO CONSOLIDADO")
print("=" * 70)

print(f"Cursos históricos : {len(indice)}")
print(f"Cursos canônicos  : {len(catalogo)}")

print()

print("Maiores grupos")

print("-" * 70)

for curso, variantes in sorted(
    catalogo.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:15]:

    print(f"{len(variantes):>2}  {curso}")

CATÁLOGO CONSOLIDADO
Cursos históricos : 132
Cursos canônicos  : 90

Maiores grupos
----------------------------------------------------------------------
 3  Artes Cênicas (Bacharelado)
 3  Biologia (Licenciatura)
 3  Ciências Sociais – Antropologia/ Sociologia (Bacharelado/Licenciatura)
 3  Design – Programação Visual/Projeto do Produto (Bacharelado)
 3  Letras - Francês (Bacharelado/Licenciatura)
 3  Letras - Inglês (Bacharelado/Licenciatura)
 3  Letras - Português (Bacharelado/Licenciatura)
 3  Matemática (Bacharelado/Licenciatura)
 3  Música (Licenciatura)
 2  Arquivologia (Bacharelado)
 2  Artes Cênicas (Literatura)
 2  Artes Visuais (Bacharelado)
 2  Artes Visuais (Licenciatura)
 2  Ciências Ambientais (Bacharelado)
 2  Ciências Contábeis (Bacharelado)


In [4]:
# ============================================================
# EXPORTAÇÃO
# ============================================================

# CSV

catalogo_csv = pd.DataFrame(

    [
        {
            "Curso canônico": canonico,
            "Curso histórico": historico
        }

        for canonico, variantes in catalogo.items()

        for historico in variantes
    ]

)

catalogo_csv.to_csv(

    DIR_CATALOGOS / "catalogo_cursos.csv",

    index=False,

    encoding="utf-8-sig"

)

# JSON (catálogo)

with open(

    DIR_CATALOGOS / "catalogo_cursos.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        catalogo,

        f,

        ensure_ascii=False,

        indent=4

    )

# JSON (índice)

with open(

    DIR_CATALOGOS / "indice_cursos.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        indice,

        f,

        ensure_ascii=False,

        indent=4

    )

print("=" * 70)
print("ARQUIVOS GERADOS")
print("=" * 70)

print("✓ catalogo_cursos.csv")
print("✓ catalogo_cursos.json")
print("✓ indice_cursos.json")

ARQUIVOS GERADOS
✓ catalogo_cursos.csv
✓ catalogo_cursos.json
✓ indice_cursos.json
